# Evaluating RAG Pipelines (RAGAS Metrics)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/06_GenAI_LLM_RAG/rag_evaluation_ragas.ipynb)

A RAG system can fail silently: bad retrieval starves the LLM; a fluent LLM can still hallucinate over good context. Evaluation needs per-stage metrics.

We compute **faithfulness**, **context relevance** and **answer relevance** from first principles on a toy case, then show the industry-standard `ragas` package for production use.

## 1. A tiny RAG run to dissect

In [ ]:
question = "What is the refund window for courses?"
retrieved_contexts = [
    "Full refunds are available within 30 days of purchase.",      # relevant
    "Courses include lifetime access to recorded sessions.",       # irrelevant
]
ground_truth   = "Students can request a full refund within 30 days."
generated_answer = "You can get your money back within 30 days."

## 2. Faithfulness - is the answer grounded in context?

In [ ]:
import re, itertools

def claims(text):
    return [c.strip() for c in re.split(r"[.;]", text) if c.strip()]

def token_set(s):
    return set(re.findall(r"\w+", s.lower()))

ctx_tokens = token_set(" ".join(retrieved_contexts))
supported, total = 0, 0
for c in claims(generated_answer):
    total += 1
    overlap = len(token_set(c) & ctx_tokens) / max(len(token_set(c)), 1)
    supported += overlap >= 0.6
    print(f"'{c}' grounded={overlap:.0%}")
print(f"\nFaithfulness ~ {supported}/{total} = {supported/total:.2f}  (1.0 = every claim traceable)")

## 3. Context precision/recall vs ground truth

In [ ]:
gt_tokens = token_set(ground_truth)

hits = [len(token_set(c) & gt_tokens) > 0 for c in retrieved_contexts]
precision_at_k = sum(hits) / len(hits)
recall = any(hits)                       # did ANY context contain it?

print(f"context precision@2 : {precision_at_k:.2f}   <- wasted slot on ctx#2")
print(f"context recall      : {recall}")

## 4. Answer relevance - does it address the question?

In [ ]:
q_tokens = token_set(question)
a_overlap = len(token_set(generated_answer) & q_tokens) / len(token_set(generated_answer))
print(f"lexical answer-relevance proxy: {a_overlap:.2f}")
print("(real systems: generate questions FROM the answer, embed, compare to original - see ragas)")

## 5. Production shortcut: the `ragas` library

In [ ]:
# Ragas scores faithfulness/answer_relevancy/context_* using an LLM as judge.
# Works with a FREE local model via Ollama (see ollama_local_llm.ipynb):

ragas_reference = """
from ragas import evaluate
from ragas.metrics import (faithfulness, answer_relevancy,
                           context_precision, context_recall)
from datasets import Dataset

ds = Dataset.from_dict({
    "question": [question],
    "answer": [generated_answer],
    "contexts": [retrieved_contexts],
    "ground_truth": [ground_truth],
})
score = evaluate(ds, metrics=[faithfulness, answer_relevancy,
                              context_precision, context_recall])
"""
print(ragas_reference)

**Debug rule of thumb**
| Symptom | Broken stage | Fix |
|---|---|---|
| answer wrong + contexts wrong | retrieval | better chunks/embeddings/top-k |
| contexts right + answer wrong | generation | stronger model / stricter prompt |
| both fine, user unhappy | eval design | add real user questions to test set |

Track these metrics on every prompt/embedding change - RAG regresses quietly.